In [ ]:
!git clone https://github.com/jburroni/IntroPPLs26.git

%cd IntroPPLs26/notebooks/Jun-19

# Activity 1 - Complete the FOPPL evaluator

A FOPPL program is parsed into nested Python lists (`Symbol` atoms for identifiers).
This notebook holds the evaluator from the lecture, one recursive function, with the
`if` case left blank. Complete it, then run the test at the bottom.

In [ ]:
import os, sys

# Locate the repo's interpreters/ folder by walking up from the working directory,
# so this notebook runs from wherever you open it.
_p = os.path.abspath(os.getcwd())
while _p != os.path.dirname(_p):
    if os.path.isdir(os.path.join(_p, "interpreters", "minippl")):
        sys.path.insert(0, os.path.join(_p, "interpreters"))
        break
    _p = os.path.dirname(_p)

import numpy as np
from minippl import parse_one, PRIMITIVES, Symbol

## The evaluator

`evaluate(expr, env, state)` walks the parsed program. `env` maps variables to values;
`state` is the inference state, holding the random-number generator and the running
`log_w`. Four forms are special: `sample` draws from the prior and moves on, `observe`
adds the datum's log-probability to `log_w` and returns the value, `let` binds
variables, and `if` chooses a branch. Everything else is a primitive call.

**Fill in the `if` case below.**

In [ ]:
class State:
    """The inference state: a random-number generator and the running log-weight."""
    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)
        self.log_w = 0.0


def evaluate(expr, env, state):
    if isinstance(expr, Symbol):                 # variable: look it up
        return env[expr]
    if not isinstance(expr, list):               # constant: number, bool, string, nil
        return expr
    op, *args = expr
    if op == "let":                              # (let [v e ...] body)
        binds, *body = args
        env = dict(env)
        for i in range(0, len(binds), 2):
            env[binds[i]] = evaluate(binds[i + 1], env, state)
        return [evaluate(e, env, state) for e in body][-1]
    if op == "if":                               # (if test then else)
        test, then, els = args
        cond_res = evaluate(test, env, state)
        if cond_res:
            return evaluate(then, env, state)
        return evaluate(els, env, state)
    if op == "sample":                           # (sample d): draw from the prior, move on
        return evaluate(args[0], env, state).sample(state.rng)
    if op == "observe":                          # (observe d v): score the datum, keep its value
        dist  = evaluate(args[0], env, state)
        value = evaluate(args[1], env, state)
        state.log_w += dist.log_prob(value)
        return value
    fn = PRIMITIVES[op] if op in PRIMITIVES else evaluate(op, env, state)
    return fn(*[evaluate(a, env, state) for a in args])


def run(src, seed=0):
    """One forward execution. Returns (value, log_weight)."""
    st = State(seed)
    return evaluate(parse_one(src), {}, st), st.log_w

## Test it

Run the cell below. Until you complete the `if` case it raises `NotImplementedError`.
Once your line is correct, likelihood weighting on the conjugate model
$m \sim \mathcal{N}(0,1)$, $y = 2$ recovers the posterior mean
$\mathbb{E}[m \mid y] = 1$.

In [ ]:
# the if case must be lazy: the branch not taken is never evaluated
assert run("(if (> 1 0) 10 (/ 1 0))")[0] == 10
assert run("(if (< 1 0) (/ 1 0) 42)")[0] == 42
print("if case OK: lazy evaluation works")

# end to end: likelihood weighting recovers the conjugate posterior mean
program = "(let [m (sample (normal 0 1))] (observe (normal m 1) 2.0) m)"

xs, lws = zip(*(run(program, seed=s) for s in range(20000)))
w = np.exp(np.array(lws) - np.max(lws))
w /= w.sum()
print("posterior mean E[m | y=2]:", round(float(np.array(xs) @ w), 3), "  (exact 1.0)")

## Stuck?

`if` is *lazy*: evaluate the test, then evaluate **only** the branch it selects, in the
same `env` and `state`. If you evaluate both branches you fire `sample` and `observe`
on the path the run never took, inventing random choices and likelihood factors that
should not exist.